# ListenBrainz Star Schema Data Quality Checks

This notebook validates the **dbt star schema tables** in BigQuery.

## Tables checked

- `dim_user`
- `dim_track`
- `dim_artist`
- `dim_album`
- `dim_date`
- `dim_time`
- `fact_listening_events`

## Main checks

1. Dimension primary keys are not null and unique.
2. Fact table `listen_id` is not null and unique.
3. Fact table foreign keys are not null.
4. Fact table foreign keys match valid dimension records.
5. Business rules such as `listen_count = 1` are valid.
6. Date and time keys are valid.


## 1. Install required packages

Run this cell if the packages are not already installed.


In [1]:
!pip install google-cloud-bigquery pandas db-dtypes


## 2. Configure BigQuery connection

Update the service account key path if needed.

For Cloud Run, you should use the service account attached to Cloud Run.  
For local Jupyter or Colab, set `GOOGLE_APPLICATION_CREDENTIALS`.


In [2]:
import os
from pathlib import Path

import pandas as pd
from google.cloud import bigquery

# Change this path to your actual service account JSON key path
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"/home/sansa/my-project-sssint1-7e02e9078e06.json"

PROJECT_ID = "my-project-sssint1"
DATASET = "listenbrainz_gcp"

client = bigquery.Client(project=PROJECT_ID)

OUTPUT_DIR = Path("listenbrainz_star_schema_dq_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project:", PROJECT_ID)
print("Dataset:", DATASET)
print("Output folder:", OUTPUT_DIR.resolve())


Project: my-project-sssint1
Dataset: listenbrainz_gcp
Output folder: /home/sansa/SCTP_DS3F_Module2_Assignment_Team7/notebooks/listenbrainz_star_schema_dq_outputs


## 3. Helper function

This function runs a BigQuery data quality check and returns pass/fail result.


In [3]:
def run_dq_check(check_name: str, sql: str) -> dict:
    """Run a data quality SQL check.

    The SQL must return one row with a column named `invalid_rows`.
    A check passes when invalid_rows == 0.
    """
    df = client.query(sql).to_dataframe()
    invalid_rows = int(df.loc[0, "invalid_rows"])
    status = "PASS" if invalid_rows == 0 else "FAIL"

    return {
        "check_name": check_name,
        "invalid_rows": invalid_rows,
        "status": status
    }


## 4. Check row counts

This confirms that all star schema tables exist and contain records.


In [4]:
row_count_sql = f"""
SELECT 'dim_user' AS table_name, COUNT(*) AS total_rows
FROM `{PROJECT_ID}.{DATASET}.dim_user`

UNION ALL
SELECT 'dim_track' AS table_name, COUNT(*) AS total_rows
FROM `{PROJECT_ID}.{DATASET}.dim_track`

UNION ALL
SELECT 'dim_artist' AS table_name, COUNT(*) AS total_rows
FROM `{PROJECT_ID}.{DATASET}.dim_artist`

UNION ALL
SELECT 'dim_album' AS table_name, COUNT(*) AS total_rows
FROM `{PROJECT_ID}.{DATASET}.dim_album`

UNION ALL
SELECT 'dim_date' AS table_name, COUNT(*) AS total_rows
FROM `{PROJECT_ID}.{DATASET}.dim_date`

UNION ALL
SELECT 'dim_time' AS table_name, COUNT(*) AS total_rows
FROM `{PROJECT_ID}.{DATASET}.dim_time`

UNION ALL
SELECT 'fact_listening_events' AS table_name, COUNT(*) AS total_rows
FROM `{PROJECT_ID}.{DATASET}.fact_listening_events`
"""

row_counts = client.query(row_count_sql).to_dataframe()
row_counts


/home/sansa/miniconda3/envs/elt/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,total_rows
0,dim_time,1440
1,fact_listening_events,146914202
2,dim_user,2684
3,dim_date,4914
4,dim_album,2505436
5,dim_artist,1106614
6,dim_track,16363969


## 5. Define star schema data quality checks

The checks below validate dimension keys, fact keys, and referential integrity.


In [5]:
dq_checks = {
    # Dimension primary key checks
    "dim_user.user_id is not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.dim_user`
        WHERE user_id IS NULL
    """,

    "dim_user.user_id is unique": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM (
            SELECT user_id
            FROM `{PROJECT_ID}.{DATASET}.dim_user`
            GROUP BY user_id
            HAVING COUNT(*) > 1
        )
    """,

    "dim_track.track_id is not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.dim_track`
        WHERE track_id IS NULL
    """,

    "dim_track.track_id is unique": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM (
            SELECT track_id
            FROM `{PROJECT_ID}.{DATASET}.dim_track`
            GROUP BY track_id
            HAVING COUNT(*) > 1
        )
    """,

    "dim_artist.artist_id is not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.dim_artist`
        WHERE artist_id IS NULL
    """,

    "dim_artist.artist_id is unique": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM (
            SELECT artist_id
            FROM `{PROJECT_ID}.{DATASET}.dim_artist`
            GROUP BY artist_id
            HAVING COUNT(*) > 1
        )
    """,

    "dim_album.album_id is not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.dim_album`
        WHERE album_id IS NULL
    """,

    "dim_album.album_id is unique": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM (
            SELECT album_id
            FROM `{PROJECT_ID}.{DATASET}.dim_album`
            GROUP BY album_id
            HAVING COUNT(*) > 1
        )
    """,

    "dim_date.date_id is not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.dim_date`
        WHERE date_id IS NULL
    """,

    "dim_date.date_id is unique": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM (
            SELECT date_id
            FROM `{PROJECT_ID}.{DATASET}.dim_date`
            GROUP BY date_id
            HAVING COUNT(*) > 1
        )
    """,

    "dim_time.time_id is not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.dim_time`
        WHERE time_id IS NULL
    """,

    "dim_time.time_id is unique": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM (
            SELECT time_id
            FROM `{PROJECT_ID}.{DATASET}.dim_time`
            GROUP BY time_id
            HAVING COUNT(*) > 1
        )
    """,

    # Fact table primary key checks
    "fact_listening_events.listen_id is not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events`
        WHERE listen_id IS NULL
    """,

    "fact_listening_events.listen_id is unique": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM (
            SELECT listen_id
            FROM `{PROJECT_ID}.{DATASET}.fact_listening_events`
            GROUP BY listen_id
            HAVING COUNT(*) > 1
        )
    """,

    # Fact foreign key null checks
    "fact foreign keys are not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events`
        WHERE user_id IS NULL
           OR track_id IS NULL
           OR artist_id IS NULL
           OR date_id IS NULL
           OR time_id IS NULL
    """,

    # Referential integrity checks
    "fact.user_id exists in dim_user": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events` f
        LEFT JOIN `{PROJECT_ID}.{DATASET}.dim_user` d
            ON f.user_id = d.user_id
        WHERE f.user_id IS NOT NULL
          AND d.user_id IS NULL
    """,

    "fact.track_id exists in dim_track": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events` f
        LEFT JOIN `{PROJECT_ID}.{DATASET}.dim_track` d
            ON f.track_id = d.track_id
        WHERE f.track_id IS NOT NULL
          AND d.track_id IS NULL
    """,

    "fact.artist_id exists in dim_artist": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events` f
        LEFT JOIN `{PROJECT_ID}.{DATASET}.dim_artist` d
            ON f.artist_id = d.artist_id
        WHERE f.artist_id IS NOT NULL
          AND d.artist_id IS NULL
    """,

    "fact.album_id exists in dim_album when not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events` f
        LEFT JOIN `{PROJECT_ID}.{DATASET}.dim_album` d
            ON f.album_id = d.album_id
        WHERE f.album_id IS NOT NULL
          AND d.album_id IS NULL
    """,

    "fact.date_id exists in dim_date": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events` f
        LEFT JOIN `{PROJECT_ID}.{DATASET}.dim_date` d
            ON f.date_id = d.date_id
        WHERE f.date_id IS NOT NULL
          AND d.date_id IS NULL
    """,

    "fact.time_id exists in dim_time": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events` f
        LEFT JOIN `{PROJECT_ID}.{DATASET}.dim_time` d
            ON f.time_id = d.time_id
        WHERE f.time_id IS NOT NULL
          AND d.time_id IS NULL
    """,

    # Business rules
    "listen_count equals 1": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events`
        WHERE listen_count != 1
           OR listen_count IS NULL
    """,

    "listened_at is not in the future": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.fact_listening_events`
        WHERE listened_at > CURRENT_TIMESTAMP()
    """,

    "dim_track.track_name is not blank": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.dim_track`
        WHERE track_name IS NULL
           OR TRIM(track_name) = ''
    """,

    "dim_artist.artist_name is not blank": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.dim_artist`
        WHERE artist_name IS NULL
           OR TRIM(artist_name) = ''
    """,

    "dim_album.album_id is not null": f"""
        SELECT COUNT(*) AS invalid_rows
        FROM `{PROJECT_ID}.{DATASET}.dim_album`
        WHERE album_id IS NULL
    """
}


## 6. Run all data quality checks

A check passes when `invalid_rows = 0`.


In [6]:
dq_results = []

for check_name, sql in dq_checks.items():
    result = run_dq_check(check_name, sql)
    dq_results.append(result)

dq_results_df = pd.DataFrame(dq_results)

# Save output
output_path = OUTPUT_DIR / "star_schema_data_quality_results.csv"
dq_results_df.to_csv(output_path, index=False)

dq_results_df


/home/sansa/miniconda3/envs/elt/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,check_name,invalid_rows,status
0,dim_user.user_id is not null,0,PASS
1,dim_user.user_id is unique,0,PASS
2,dim_track.track_id is not null,0,PASS
3,dim_track.track_id is unique,0,PASS
4,dim_artist.artist_id is not null,0,PASS
5,dim_artist.artist_id is unique,0,PASS
6,dim_album.album_id is not null,0,PASS
7,dim_album.album_id is unique,0,PASS
8,dim_date.date_id is not null,0,PASS
9,dim_date.date_id is unique,0,PASS


## 7. Show failed checks only

If this table is empty, all checks passed.


In [7]:
failed_checks = dq_results_df[dq_results_df["status"] == "FAIL"]
failed_checks


,check_name,invalid_rows,status


## 8. Investigate duplicate fact listen IDs

Run this only if `fact_listening_events.listen_id is unique` fails.


In [8]:
duplicate_fact_listen_ids_sql = f"""
SELECT
    listen_id,
    COUNT(*) AS duplicate_count
FROM `{PROJECT_ID}.{DATASET}.fact_listening_events`
GROUP BY listen_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20
"""

duplicate_fact_listen_ids = client.query(duplicate_fact_listen_ids_sql).to_dataframe()
duplicate_fact_listen_ids


/home/sansa/miniconda3/envs/elt/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,listen_id,duplicate_count


## 9. Investigate duplicate track IDs

Run this only if `dim_track.track_id is unique` fails.


In [9]:
duplicate_track_ids_sql = f"""
SELECT
    track_id,
    COUNT(*) AS duplicate_count
FROM `{PROJECT_ID}.{DATASET}.dim_track`
GROUP BY track_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20
"""

duplicate_track_ids = client.query(duplicate_track_ids_sql).to_dataframe()
duplicate_track_ids


/home/sansa/miniconda3/envs/elt/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,track_id,duplicate_count


## 10. Summary table for report

This creates a grouped summary of pass/fail checks.


In [10]:
summary = (
    dq_results_df
    .groupby("status")
    .size()
    .reset_index(name="number_of_checks")
)

summary


,status,number_of_checks
0,PASS,25


## 11. Report wording

You can use the text below in your report.

> Data quality checks were performed on the ListenBrainz star schema in BigQuery. The checks validated primary key uniqueness in all dimension tables, uniqueness of the fact table listening event ID, non-null foreign keys, referential integrity between the fact table and dimensions, and business rules such as valid listening timestamps and listen count values. These checks help ensure that the star schema is reliable for downstream business analysis, dashboards, and machine learning enhancements.
